# SIGReg on Self-Flow token embeddings

Self-Flow (Chefer et al., 2026) trains a SiT with **Dual-Timestep Scheduling** plus an **EMA teacher**. The student sees mixed noise levels; the teacher sees the cleaner time and the student predicts its features. That teacher/EMA pair is a JEPA mechanism: stop-gradient, slow targets, collapse prevention.

LeJEPA replaces those heuristics with **SIGReg**: force embeddings toward an isotropic Gaussian $N(0, I)$ with a sliced Epps–Pulley test. The question in this kata is whether we can put SIGReg on the SiT’s own token embeddings and drop the teacher, the EMA copy, and any external encoder (DINO / REPA).

## Verdict

**Yes as an anti-collapse term, no as a full substitute for the SSL task.**

The EMA teacher in Self-Flow does two jobs:

1. **Collapse prevention** (stop-grad + slow-moving targets). This is the JEPA hack. SIGReg is a better, theory-backed replacement for *this* job.
2. **Target construction.** Dual-timestep creates an information asymmetry; the teacher encodes the *cleaner* view that the student must predict. That is the actual representation-learning signal. Gaussianity of features is a geometric prior, not a semantic one.

External encoders (REPA + DINO) supply semantics. SIGReg supplies a well-conditioned embedding distribution. They are not interchangeable. The version of the idea that can actually replace the teacher is:

- keep Dual-Timestep Scheduling (it already helps generation on its own),
- drop the EMA copy,
- predict cleaner-view features with the *same* SiT,
- put SIGReg on those token embeddings so the predictive loss cannot collapse.

Just adding `λ · SIGReg(tokens)` on top of vanilla flow matching will un-collapse features, but it will not invent DINO-like semantics by itself.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pyplot as plt
import torch

from pytorch_katas.metrics import collapse_metrics
from pytorch_katas.self_flow import Recipe, SelfFlowTrainer, euler_sample
from pytorch_katas.sigreg import sigreg_loss
from pytorch_katas.sit import MiniSiT

## SIGReg in 2D

Project embeddings onto random unit directions, match each 1-D slice to $N(0,1)$ with the Epps–Pulley characteristic-function test, average the slices. Minimized at isotropic Gaussian. Exact collapse is a symmetry (identical vectors get identical gradients), so we start from a thin Gaussian cloud rather than a single point.

In [ ]:
torch.manual_seed(0)
z = torch.tensor([2.0, 0.0]) + torch.randn(400, 2) * torch.tensor([1.5, 0.05])
z = z.clone().requires_grad_(True)
opt = torch.optim.SGD([z], lr=1.0)
traj = [z.detach().clone()]
for _ in range(80):
    opt.zero_grad()
    sigreg_loss(z, num_slices=128).backward()
    opt.step()
    traj.append(z.detach().clone())

fig, axes = plt.subplots(1, 2, figsize=(8, 4), sharex=True, sharey=True)
for ax, pts, title in zip(axes, (traj[0], traj[-1]), ("before SIGReg", "after SIGReg")):
    ax.scatter(pts[:, 0], pts[:, 1], s=8, alpha=0.5)
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
print("loss", float(sigreg_loss(traj[0])), "->", float(sigreg_loss(traj[-1])))
print("std ", float(traj[0].std()), "->", float(traj[-1].std()))

## Four recipes on a tiny SiT

| recipe | dual-timestep | teacher / EMA | extra loss |
|---|---|---|---|
| `vanilla` | no | no | flow matching only |
| `self_flow` | yes | yes | cosine align to EMA teacher |
| `sigreg` | yes | **no** | SIGReg on student token embeddings |
| `lejepa_flow` | yes | **no** | predict cleaner-view features **+** SIGReg |

`sigreg` is the literal proposal (Gaussianize the SiT’s own tokens). `lejepa_flow` is the version that can actually replace JEPA: keep the predictive task, drop EMA, use SIGReg as anti-collapse.

Apply SIGReg to flattened mid-layer tokens `(B·N, D)`, not to the velocity head. High-`t` tokens are already noise-like; Dual-Timestep still exposes a cleaner view, which is where you want the Gaussian prior.

In [ ]:
def make_trainer(recipe: Recipe) -> SelfFlowTrainer:
    model = MiniSiT(image_size=16, patch_size=4, dim=64, n_heads=4, n_layers=4, n_classes=10)
    return SelfFlowTrainer(
        model,
        recipe,
        student_layer=1,
        teacher_layer=3,
        gamma=0.5,
        lambd=0.05,
        sigreg_slices=64,
    )


images = torch.rand(8, 3, 16, 16) * 2 - 1
y = torch.randint(0, 10, (8,))

for recipe in Recipe:
    trainer = make_trainer(recipe)
    losses = trainer.training_step(images, y)
    losses.total.backward()
    trainer.after_optimizer_step()
    print(
        f"{recipe.value:12s}  teacher={trainer.teacher is not None}"
        f"  gen={losses.gen.item():.3f}  rep={losses.rep.item():.3f}  sigreg={losses.sigreg.item():.3f}"
    )

## A few SGD steps: does SIGReg actually spread SiT tokens?

On random images this is not a generation benchmark. It only checks that the regularizer is live in the SiT feature stream and that the teacher-free recipes do not need an EMA module.

In [ ]:
def feature_snapshot(trainer: SelfFlowTrainer, images: torch.Tensor, y: torch.Tensor) -> dict[str, float]:
    trainer.model.eval()
    with torch.no_grad():
        x0 = trainer.model.patchify(images)
        t = torch.zeros(images.shape[0], trainer.model.n_tokens)
        out = trainer.model(x0, t, y, return_layers=(trainer.student_layer,))
    return collapse_metrics(out.features[trainer.student_layer])


def train_briefly(recipe: Recipe, steps: int = 20) -> dict[str, float]:
    trainer = make_trainer(recipe)
    opt = torch.optim.AdamW(trainer.trainable_parameters(), lr=1e-3)
    before = feature_snapshot(trainer, images, y)
    trainer.train()
    for _ in range(steps):
        opt.zero_grad()
        losses = trainer.training_step(images, y)
        losses.total.backward()
        opt.step()
        trainer.after_optimizer_step()
    after = feature_snapshot(trainer, images, y)
    print(recipe.value, "std", f"{before['std']:.3f}->{after['std']:.3f}", "abs_cos", f"{before['abs_cosine']:.3f}->{after['abs_cosine']:.3f}")
    return after


for recipe in (Recipe.VANILLA, Recipe.SIGREG, Recipe.LEJEPA_FLOW):
    train_briefly(recipe)

## Sampling

Inference is ordinary homogeneous-time flow matching. Dual-timestep and SIGReg are training-only. Untrained samples are noise; this cell only checks the Euler path.

In [ ]:
model = MiniSiT(image_size=16, patch_size=4, dim=64, n_heads=4, n_layers=4, n_classes=10)
samples = euler_sample(model, torch.zeros(4, dtype=torch.long), steps=8)
print(samples.shape)
grid = samples.permute(0, 2, 3, 1).mul(0.5).add(0.5).clamp(0, 1).reshape(2, 2, 16, 16, 3).permute(0, 2, 1, 3, 4).reshape(32, 32, 3)
plt.figure(figsize=(3, 3))
plt.imshow(grid.cpu())
plt.axis("off")
plt.title("untrained Euler samples")

## What I would actually try next

1. **Do not drop Dual-Timestep.** Fig. 2b in Self-Flow already shows it helps generation without any teacher. SIGReg does not replace that asymmetry.
2. **Prefer `lejepa_flow` over bare `sigreg`.** Bare SIGReg Gaussianizes whatever features the flow loss already learned. The predictive loss is what forces global inference from cleaner tokens.
3. **Regularize a mid layer, on the cleaner view.** Matching Self-Flow / REPA: student layer $l$ < teacher layer $k$. SIGReg on velocity or on $t \approx 1$ fights the generative objective.
4. **Flatten tokens as the sample axis** `(B·N, D)`. That matches “on top of token embeddings.” If spatial identity collapses, fall back to SIGReg on mean-pooled or register tokens only.
5. **Do not expect this to beat DINO at ImageNet linear probe without a predictive loss.** Gaussian + flow matching ≠ semantic clustering. The bet is that Dual-Timestep prediction + SIGReg is enough to *not need* DINO for *generation*, which is Self-Flow’s actual claim.

Code: `pytorch_katas.sigreg`, `pytorch_katas.sit`, `pytorch_katas.self_flow`.